In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score

from sklearn.model_selection import GridSearchCV

In [2]:
# Load your dataset
df = pd.read_csv("CKD.csv")

# One-hot encode categorical variables (drop_first=True to avoid dummy trap)
df = pd.get_dummies(df, drop_first=True)

# Split features and target
X = df.drop("classification_yes", axis=1)  # classification_yes = 1 if 'yes', 0 if 'no'
y = df["classification_yes"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize numeric features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [7]:
models = {
    "Logistic Regression": LogisticRegression(),
    "SVM": SVC(probability=True),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()
}
# Define parameter grids for each model
param_grids = {
    "Logistic Regression": {
        'solver': ['newton-cg', 'lbfgs', 'liblinear', 'saga'],
        'penalty': ['l2']
    },
    "SVM": {
        'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
        'gamma': ['auto', 'scale'],
        'C': [10, 100, 1000, 2000, 3000]
    },
    "Decision Tree": {
        'criterion': ['gini', 'entropy'],
        'max_features': ['auto', 'sqrt', 'log2']
    },
    "Random Forest": {
        'criterion': ['gini', 'entropy'],
        'max_features': ['auto', 'sqrt', 'log2'],
        'n_estimators': [10, 100]
    }
}

results = []

for name, model in models.items():
    print(f"Running GridSearchCV for {name}...")
    param_grid = param_grids.get(name, {})
    
    # Use GridSearchCV only if param grid is defined, else just fit
    if param_grid:
        grid = GridSearchCV(model, param_grid, scoring='f1_weighted', refit=True, n_jobs=-1, verbose=2)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_
    else:
        # No param grid defined, just fit the model
        best_model = model.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = best_model.predict(X_test)
    
    # For models that have predict_proba, get probabilities, else decision_function or skip ROC AUC
    try:
        y_proba = best_model.predict_proba(X_test)[:, 1]
        roc = roc_auc_score(y_test, y_proba)
    except AttributeError:
        try:
            # For SVM without predict_proba, use decision_function and convert to probabilities
            y_scores = best_model.decision_function(X_test)
            roc = roc_auc_score(y_test, y_scores)
        except AttributeError:
            roc = None
    
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        "Model": name,
        "F1 Score (weighted)": round(f1, 3),
        "ROC AUC Score": round(roc, 3) if roc is not None else "N/A"
    })

# Print results
for res in results:
    print(res)


Running GridSearchCV for Logistic Regression...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Running GridSearchCV for SVM...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Running GridSearchCV for Decision Tree...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
10 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_par

Running GridSearchCV for Random Forest...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
20 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
6 fits failed with the following error:
Traceback (most recent call last):
  File "C:\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\anaconda3\Lib\site-packages\sklearn\base.py", line 1466, in wrapper
    estimator._validate_params()
  File "C:\anaconda3\Lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "C:\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 95, in validate_par

{'Model': 'Logistic Regression', 'F1 Score (weighted)': 0.987, 'ROC AUC Score': 0.998}
{'Model': 'SVM', 'F1 Score (weighted)': 0.987, 'ROC AUC Score': 0.998}
{'Model': 'Decision Tree', 'F1 Score (weighted)': 0.95, 'ROC AUC Score': 0.949}
{'Model': 'Random Forest', 'F1 Score (weighted)': 1.0, 'ROC AUC Score': 1.0}


In [8]:
results_df = pd.DataFrame(results)
print(results_df)

                 Model  F1 Score (weighted)  ROC AUC Score
0  Logistic Regression                0.987          0.998
1                  SVM                0.987          0.998
2        Decision Tree                0.950          0.949
3        Random Forest                1.000          1.000
